# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zoye-J/FlyRank--MachineLearning/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
import os
import pandas as pd
import numpy as np
import duckdb

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("CREATE SECRET hf (TYPE huggingface, PROVIDER credential_chain);")

FACT_MAR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
FACT_APR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"
FACT_MAY = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/*.parquet"
DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

print("Connected to warehouse.")

Connected to warehouse.


## 1. Two paper findings + my methodology questions


I picked two findings from the FlyRank March 2026 paper that sit closest to my lane (search intent, visibility, anomaly-shaped pages). For each, one methodology question, framed the way I'd want my own Week 5 model reviewed.

- Finding 6  "AI Traffic: A Different Signal" :
The paper reports that high-AI content averages ~9× more impressions than no-AI content, but a *weaker* Google position. AI referrals are 1.06% of tracked sessions (17,344 of 1.6M).



Methodology question: where does the AI label come from, and does the validation design carry the "different profile" claim at this prevalence?
The "ai_sessions" bucket is built from known referral rules, so the label is defined by which providers the tracking recognizes, not by an observed behavior. At 1.06% of sessions, the high_ai bucket (n=873) is small enough that a handful of large pages could drive the whole comparison. The paper itself calls the conclusion narrow. A stronger version of this finding would show the profile split survives after controlling for impression tier, or would report the bucket medians rather than means.

- ML Appendix "The Content Archetypes" :
The paper runs k-means (k=5) on the sampled active-content set. Three of the five clusters carry the same heuristic label (Rising Stars) in the original script; the paper discloses this and recommends comparing cluster profiles rather than treating them as operational personas.

Methodology question: if cluster labels were heuristic and duplicated, what does the cluster structure actually validate? k=5 was chosen, not tested against alternatives, the paper does not report a silhouette score, an elbow plot, or stability across seeds. Two of the clusters (1 and 2) differ mainly by impressions (37.9K vs 928) and the rest sit within 39–48 health. That reads more like a visibility gradient than five distinct archetypes. The methodology question is: would the same five clusters appear if k were swept, or are they an artifact of a fixed k?

Why these two:

Both are places where the paper's own disclosure already names the limitation, which makes the questions constructive rather than adversarial. Both are also directly relevant to my lane: Finding 6 shares the visibility-vs-position tension I saw in Week 5, and the Archetypes appendix is the paper's closest analogue to th epages that don't fit the expected shape, which is the honest version of my security/anomaly motivation.

## 2. My model under an honest split (before/after)


My Week-5 model (Logistic Regression on the same five features from ML-08) was already trained under a **grouped split by 'client_hash_id' that's the honest design and it was in place from the first run.
So the before/after I can show is in-sample vs held-out on the same grouped split:

- Before (in-sample): the model scored on the training rows. This is the optimistic number that a leaky or ungrouped design would report.
- After (held-out): the same model scored on the 21.7% test set, with zero client overlap.

The gap between them is the model's actual generalisation gap. If the in-sample number looks much better than the held-out number, the model has memorised its training clients and the held-out number is the one to keep.

I did not run a random split for comparison, because a random split would share clients between train and test and produce a fake lift which would measure memorisation, not skill. The grouped split is the only honest axis here.

In [3]:
# Rebuilding the modeling frame exactly as in ML08
MODEL_Q = f"""
WITH perf AS (
  SELECT
    content_hash_id,
    ANY_VALUE(client_hash_id)         AS client_hash_id,
    SUM(gsc_impressions)              AS impressions_30d,
    SUM(gsc_clicks)                   AS clicks_30d,
    AVG(NULLIF(gsc_avg_position, 0))  AS avg_position_30d
  FROM '{FACT_MAR}'
  GROUP BY content_hash_id
),
future AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS imp_apr_may
  FROM (
    SELECT content_hash_id, gsc_impressions FROM '{FACT_APR}'
    UNION ALL
    SELECT content_hash_id, gsc_impressions FROM '{FACT_MAY}'
  )
  GROUP BY content_hash_id
)
SELECT
  p.content_hash_id,
  p.client_hash_id,
  p.impressions_30d,
  p.clicks_30d,
  p.avg_position_30d,
  d.content_type,
  d.main_intent,
  d.word_count,
  DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') AS days_since_update,
  f.imp_apr_may
FROM perf p
LEFT JOIN '{DIM_CONTENT}' d ON p.content_hash_id = d.content_hash_id
LEFT JOIN future f ON p.content_hash_id = f.content_hash_id
WHERE p.impressions_30d IS NOT NULL AND p.impressions_30d > 0
"""
df = con.execute(MODEL_Q).df()

df["ctr_30d"] = np.where(df["impressions_30d"] > 0,
                         100.0 * df["clicks_30d"] / df["impressions_30d"],
                         np.nan)
df["is_declining_future"] = np.where(
    df["imp_apr_may"].isna() | (df["impressions_30d"] == 0),
    np.nan,
    (df["imp_apr_may"] < 0.8 * df["impressions_30d"]).astype(float),
)
df = df.dropna(subset=["is_declining_future"]).copy()
print(f"Modeling frame: {len(df):,} rows, base rate {df['is_declining_future'].mean():.3f}")

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))
train_df = df.iloc[train_idx].copy()
test_df  = df.iloc[test_idx].copy()

overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
assert len(overlap) == 0, "GROUP LEAK"
print(f"Train {len(train_df):,} rows / {train_df['client_hash_id'].nunique()} clients")
print(f"Test  {len(test_df):,} rows / {test_df['client_hash_id'].nunique()} clients")
print(f"Client overlap: {len(overlap)}")

FEATURES = ["impressions_30d", "clicks_30d", "ctr_30d", "avg_position_30d", "days_since_update"]

def make_X(d):
    X = d[FEATURES].copy()
    X["avg_position_30d"] = X["avg_position_30d"].fillna(-1)
    X["days_since_update"] = X["days_since_update"].fillna(-1)
    X["ctr_30d"] = X["ctr_30d"].fillna(0)
    return X

X_train, X_test = make_X(train_df), make_X(test_df)
y_train = train_df["is_declining_future"].values
y_test  = test_df["is_declining_future"].values

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_train_s, y_train)

train_scores = lr.predict_proba(X_train_s)[:, 1]
test_scores  = lr.predict_proba(X_test_s)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

print()
print("LR under the honest grouped split — before/after:")
print(f"{'':>10} {'P@20':>8} {'P@50':>8} {'P@100':>8}")
print(f"{'In-sample':>10} {precision_at_k(train_scores, y_train, 20):>8.3f} "
      f"{precision_at_k(train_scores, y_train, 50):>8.3f} "
      f"{precision_at_k(train_scores, y_train, 100):>8.3f}")
print(f"{'Held-out':>10} {precision_at_k(test_scores, y_test, 20):>8.3f} "
      f"{precision_at_k(test_scores, y_test, 50):>8.3f} "
      f"{precision_at_k(test_scores, y_test, 100):>8.3f}")
print()
print(f"Train base rate: {y_train.mean():.3f}")
print(f"Test  base rate: {y_test.mean():.3f}")
print()
print("The gap between in-sample and held-out is the generalisation gap.")
print("The held-out number is the one to keep.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling frame: 176,738 rows, base rate 0.283
Train 138,310 rows / 37 clients
Test  38,428 rows / 10 clients
Client overlap: 0

LR under the honest grouped split — before/after:
               P@20     P@50    P@100
 In-sample    0.450    0.580    0.620
  Held-out    0.250    0.260    0.360

Train base rate: 0.310
Test  base rate: 0.186

The gap between in-sample and held-out is the generalisation gap.
The held-out number is the one to keep.


## 3. Leakage audit


What I checked:

1. Label source disjoint from features. The label is 'is_declining_future', computed from 'imp_apr_may < 0.8 * impressions_30d'. The five features are all March-only aggregates. No overlap.
2. No future window in the features. 'imp_apr_may' appears only in the label construction, never in 'FEATURES'.
3. No pseudonym in the features.'client_hash_id' and 'content_hash_id' are used for grouping and joining only.
4. Train-without test on the top feature. My LR leans on 'clicks_30d' (standardized coefficient −0.934 in ML08). I retrain without it and watch whether the score collapses and if it does, the model has learned a single dominant signal rather than a distributed one.
5. Train-without test on the leak experiment from ML05. Add 'imp_apr_may' as a feature on purpose and confirm the score jumps toward 1.0. This proves the harness would catch a leak if one appeared.

I keep the honest number, not the leaky one.

In [5]:
# Audit 1  feature / label-source disjoint
rule_inputs = FEATURES
label_sources = ["imp_apr_may", "is_declining_future"]
leak = set(rule_inputs) & set(label_sources)
print("Audit 1  feature / label-source disjoint:")
print(f"  Features:      {rule_inputs}")
print(f"  Label sources: {label_sources}")
print(f"  Overlap:       {leak if leak else 'NONE — clean'}")
print()

# Audit 2  train without the dominant feature
FEATURES_NO_CLICKS = [f for f in FEATURES if f != "clicks_30d"]

def fit_and_score(feature_list):
    Xtr = scaler.fit_transform(make_X(train_df)[feature_list])
    Xte = scaler.transform(make_X(test_df)[feature_list])
    m = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
    m.fit(Xtr, y_train)
    return precision_at_k(m.predict_proba(Xte)[:, 1], y_test, 50)

p50_full = fit_and_score(FEATURES)
p50_no_clicks = fit_and_score(FEATURES_NO_CLICKS)

print("Audit 2  train-without test on the dominant feature (clicks_30d):")
print(f"  P@50 with clicks_30d    : {p50_full:.3f}")
print(f"  P@50 without clicks_30d : {p50_no_clicks:.3f}")
print(f"  Collapse: {p50_full - p50_no_clicks:+.3f}")
print("  Interpretation: P@50 is unchanged when clicks_30d is removed, even though its  coefficient (-0.934) was the largest")

# Audit 3  deliberately add a leaky feature, confirm the harness catches it
X_leak_train = make_X(train_df).copy()
X_leak_test  = make_X(test_df).copy()
X_leak_train["imp_apr_may_LEAK"] = train_df["imp_apr_may"].values
X_leak_test["imp_apr_may_LEAK"]  = test_df["imp_apr_may"].values

scaler_lk = StandardScaler()
Xtr_lk = scaler_lk.fit_transform(X_leak_train)
Xte_lk = scaler_lk.transform(X_leak_test)
lr_lk = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr_lk.fit(Xtr_lk, y_train)
p50_leak = precision_at_k(lr_lk.predict_proba(Xte_lk)[:, 1], y_test, 50)

print("Audit 3  deliberate leak (add imp_apr_may as a feature):")
print(f"  P@50 honest   : {p50_full:.3f}")
print(f"  P@50 with leak: {p50_leak:.3f}")
print(f"  Jump: {p50_leak - p50_full:+.3f}")
if p50_leak - p50_full > 0.30:
    print("The harness catches the leak. The honest number is the one I keep.")
else:
    print("The leak did not dominate the ranking.")

Audit 1  feature / label-source disjoint:
  Features:      ['impressions_30d', 'clicks_30d', 'ctr_30d', 'avg_position_30d', 'days_since_update']
  Label sources: ['imp_apr_may', 'is_declining_future']
  Overlap:       NONE — clean

Audit 2  train-without test on the dominant feature (clicks_30d):
  P@50 with clicks_30d    : 0.260
  P@50 without clicks_30d : 0.260
  Collapse: +0.000
  Interpretation: P@50 is unchanged when clicks_30d is removed, even though its  coefficient (-0.934) was the largest
Audit 3  deliberate leak (add imp_apr_may as a feature):
  P@50 honest   : 0.260
  P@50 with leak: 1.000
  Jump: +0.740
The harness catches the leak. The honest number is the one I keep.


## 4. Claim rewrite

My boldest sentence from ML 08 was:

> "The Search-Intent hypothesis that visible low-CTR pages are the right review target is not supported by what a linear model finds on this slice."
 "Not supported" reads as a general verdict on the hypothesis. What I actually have is one run, on one month, with one feature set, under one split.

> In this slice (March 2026 prior window, April–May 2026 label, grouped by client), a logistic regression on five raw signals scored Precision@50 = 0.26 on held-out clients below the Week-4 hand rule's 0.52 on the same test set. The model leaned on 'clicks_30d' rather than the CTR-vs-tier signal the rule uses, which suggests that on this slice a raw-signal linear model does not recover the same signal the hand rule does. This is a decision-support observation, not a verdict on the underlying hypothesis: the result could change with a larger prior window, a different feature set, or a different split.

On the security angle, the natural extension of this work is anomaly detection in content portfolios: pages that behave unusually (high impressions, weak position, or rare intent-content pairings) could merit a human look. The FlyRank paper's AI-Traffic finding and its k-means archetypes appendix are the closest the paper comes to that framing. But the paper is a content-strategy study and this notebook is a model audit,neither supports any claim about malicious content, phishing, spoofing, or black-hat SEO. I name the motivation and refuse the overclaim.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.